# User-based CF + Dot Production

Memory-Based Algorithm
- Item based
- **User based: 이번엔 dot product 버전 구현할 것**


Model-Based Algorithm
- Latent Factor 협업 필터링 방법 (Matrix Factorization)

# 구글 드라이브 연결

파일 > 드라이브 마운트 클릭

In [ ]:
import os
from pathlib import Path
DATA_DIR = Path("drive/MyDrive/kw/추천특강/강의자료/data/")
os.listdir(DATA_DIR)

# 데이터 로드

In [ ]:
MOVIE_DATA_PATH = f"{DATA_DIR}/movies_refined.csv"
RATING_DATA_PATH = f"{DATA_DIR}/ratings_refined.csv"


로드 후 데이터 타입 형변환  
id는 문자열로 처리

영화 데이터
```
movie_id: str
```

평점 데이터
```
user_id: str
movie_id: str
```

In [ ]:
import pandas as pd

movies_df = pd.read_csv(MOVIE_DATA_PATH,
    usecols=['movie_id', 'title'],
    dtype={'movie_id': str})

ratings_df = pd.read_csv(RATING_DATA_PATH,
    usecols=['user_id', 'movie_id', 'rating'],
    dtype={'user_id': str, 'movie_id': str})

In [ ]:
movie_ratings_df = pd.merge(ratings_df, movies_df, on='movie_id', how='left')
movie_ratings_df

,user_id,movie_id,rating,title
0,1,1,4.0,Toy Story (1995)
1,1,3,4.0,Grumpier Old Men (1995)
2,1,6,4.0,Heat (1995)
3,1,47,5.0,Seven (a.k.a. Se7en) (1995)
4,1,50,5.0,"Usual Suspects, The (1995)"
...,...,...,...,...
100784,610,166534,4.0,Split (2017)
100785,610,168248,5.0,John Wick: Chapter Two (2017)
100786,610,168250,5.0,Get Out (2017)
100787,610,168252,5.0,Logan (2017)


null 값 체크

In [ ]:
movie_ratings_df.columns[movie_ratings_df.isna().any()].tolist()

[]

영화명 결측치 체크

In [ ]:
movie_ratings_df[movie_ratings_df['title'].isnull()]

,user_id,movie_id,rating,title


# 유저 유사도 행렬 준비


유저X 입력되었을 때

유저X와 비슷한 유저 찾는데 사용할 행렬

In [ ]:
USER_SIMILARITY_DATA_PATH = f"{DATA_DIR}/user_similarity_with_index.csv"

In [ ]:
# 유저와 유저 간에 대한 유사도
# 인덱스 정보 있으므로 index_col 옵션 사용
user_similarity_df = pd.read_csv(USER_SIMILARITY_DATA_PATH, index_col=0)
user_similarity_df

,1,10,100,101,102,103,104,105,106,107,...,90,91,92,93,94,95,96,97,98,99
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.016875,0.147763,0.141447,0.156466,0.211480,0.121173,0.090396,0.019425,0.066894,...,0.051841,0.334727,0.028767,0.232766,0.110082,0.178069,0.282780,0.086557,0.062187,0.066231
10,0.016875,1.000000,0.090474,0.020533,0.036215,0.116766,0.178308,0.149666,0.133146,0.003137,...,0.000000,0.055251,0.016773,0.029036,0.038062,0.042865,0.018888,0.057771,0.130223,0.020587
100,0.147763,0.090474,1.000000,0.070894,0.142033,0.136122,0.178267,0.065512,0.000000,0.098179,...,0.015134,0.189074,0.000000,0.122328,0.124926,0.093664,0.071043,0.078111,0.053039,0.079063
101,0.141447,0.020533,0.070894,1.000000,0.026755,0.105728,0.045393,0.065482,0.000000,0.000000,...,0.000000,0.133919,0.000000,0.012955,0.000000,0.070774,0.107449,0.103712,0.000000,0.000000
102,0.156466,0.036215,0.142033,0.026755,1.000000,0.094133,0.087527,0.051604,0.029935,0.353838,...,0.000000,0.259438,0.000000,0.219928,0.586507,0.046217,0.136751,0.048376,0.054637,0.428520
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.178069,0.042865,0.093664,0.070774,0.046217,0.184223,0.125659,0.143146,0.044271,0.000000,...,0.061351,0.265121,0.023369,0.091787,0.054546,1.000000,0.208704,0.089793,0.054944,0.023075
96,0.282780,0.018888,0.071043,0.107449,0.136751,0.177743,0.087595,0.083982,0.021467,0.130466,...,0.044602,0.271696,0.000000,0.240489,0.204494,0.208704,1.000000,0.144190,0.057705,0.072585
97,0.086557,0.057771,0.078111,0.103712,0.048376,0.102086,0.048995,0.141523,0.075348,0.000000,...,0.000000,0.150644,0.000000,0.045750,0.044044,0.089793,0.144190,1.000000,0.080515,0.000000


pandas 주의사항

index가 0부터 시작하지 않을 경우
```
.loc[label]: 레이블(이름) 기반 인덱싱. user_id 값 사용할 때 loc 사용
.iloc[position]: iloc은 여전히 위치 기반 인덱싱. 항상 0부터 시작하므로 사용 시 주의!
```


In [ ]:
# 높은수록 비슷한 유저
# 가장 높은 유저는 본인 자신 [0] index 제외
sample_user_id = "1"
user_similarity_df[sample_user_id].sort_values(ascending=False)[1:10]

,1
user_id,
266,0.357408
313,0.351562
368,0.345127
57,0.345034
91,0.334727
469,0.330664
39,0.329782
288,0.329700
452,0.328048


# Dot Production 알고리즘

임의의 영화 m에 대해

U: 모든 사용자  
U_m: 영화m을 본 사용자 집합  
R_um: U_m이 영화m에 대해 단 평점 집합

내적 계산 대상
- 유사도 행렬: U_m과 U의 유사도
- 평점 행렬: R_um
- 정규화 상수: 유저별 U_m과의 유사도 총합


내적 방법
- 유사도행렬 x 평점 행렬 / 정규화 상수

위 내적을 모든 M마다 반복해서 계산


In [ ]:
# 테스트용 임의의 영화 선택
title = "Black Robe"

In [ ]:
# 해당 영화를 본 사용자들
watched_user_indices = (
    movie_ratings_df[movie_ratings_df['title'].str.contains(title)].index
)

watched_user_ids_str = movie_ratings_df.loc[watched_user_indices, 'user_id'].tolist()
watched_user_ids_str

['28', '140', '414']

In [ ]:
watched_user_ids = [int(user_id) for user_id in watched_user_ids_str]
watched_user_ids

[28, 140, 414]

## Dot Product: (A) 분자1

(전체 유저 x m영화본 유저) 유사도 행렬

In [ ]:
# 유사도 subset
# (행)영화본유저 x (열)전체유저
sub_similarity_df = user_similarity_df.loc[watched_user_ids]
sub_similarity_df

,1,10,100,101,102,103,104,105,106,107,...,90,91,92,93,94,95,96,97,98,99
user_id,,,,,,,,,,,,,,,,,,,,,
28,0.203089,0.101135,0.092790,0.098516,0.127745,0.245316,0.096204,0.271741,0.090382,0.077507,...,0.040468,0.296880,0.013918,0.120781,0.123806,0.233805,0.189752,0.117615,0.074620,0.071974
140,0.200202,0.100272,0.192313,0.059070,0.187521,0.218464,0.141306,0.177975,0.086919,0.161502,...,0.038658,0.307193,0.059536,0.178922,0.186524,0.175955,0.239769,0.175142,0.087512,0.110336
414,0.281998,0.144116,0.226562,0.140453,0.133455,0.322758,0.242816,0.294781,0.092636,0.096225,...,0.060622,0.381520,0.072162,0.166943,0.132443,0.250975,0.177718,0.111082,0.149007,0.105131


내적 곱 위해 행렬 전치

In [ ]:
# 전치행렬 (전체유저610명 x 이 영화 본 유저3명)
sub_similarity_matrix = sub_similarity_df.T.to_numpy()
sub_similarity_matrix

array([[0.20308944, 0.20020157, 0.28199794],
       [0.10113478, 0.10027172, 0.1441157 ],
       [0.09279042, 0.19231314, 0.22656158],
       ...,
       [0.11761516, 0.17514167, 0.11108194],
       [0.07461958, 0.08751155, 0.14900705],
       [0.0719742 , 0.1103361 , 0.10513064]])

분자 1 의 행렬 크기

전체 유저수 x 영화본 유저수

In [ ]:
sub_similarity_matrix.shape

(610, 3)

## Dot Product: (B) 분자 2

분자 1과 내적곱 시킬 행렬

이 영화 본 3명이 작성한 평점

In [ ]:
movie_ratings_df.loc[watched_user_indices]

,user_id,movie_id,rating,title
4453,28,4688,2.5,Black Robe (1991)
21460,140,4688,3.5,Black Robe (1991)
63735,414,4688,4.0,Black Robe (1991)


In [ ]:
watched_user_y = movie_ratings_df.loc[watched_user_indices, 'rating']
watched_user_y

,rating
4453,2.5
21460,3.5
63735,4.0


행렬 연산 위해 reshape

In [ ]:
import numpy as np

watched_user_y = np.array(watched_user_y.tolist()).reshape(-1, 1)
watched_user_y

array([[2.5],
       [3.5],
       [4. ]])


분자1 Χ 분자2

(전체 유저수 x 영화본유저수) Χ (영화본유저수 x 1)

이제 shape 맞아서 내적 가능

In [ ]:
watched_user_y.shape

(3, 1)

## Dot Product: (N) 분모

정규화 위해 나눠주는 값


현재 유저 한 명에 대한 영화본 유저 u_m 의 유사도 총합,
```
       X=1
28	 0.203089
140	0.200202
414	0.281998

총합 = 0.203089 + 0.200202 + 0.281998
```

전체 유저를 행렬로 한번에 계산하기 위해

전체 유저에 대한 영화본 유저 u_m 의 유사도 총합 준비


Black Robe본 유저 3명과 (28, 140, 414)

유저 1 (userId[0])과의 유사도

In [ ]:
# Black Robe
sub_similarity_matrix[0]

array([0.20308944, 0.20020157, 0.28199794])

유사도 총합 + 1 (div by 0 에러 피하기 위함)

```
(0.2074366 + 0.19928216 + 0.28574522) + 1
```

In [ ]:
sum(sub_similarity_matrix[0]) + 1

np.float64(1.6852889530733024)

위 방식을 한번에 계산

Black Robe본 유저 3명과

모든 유저와의 유사도 구한 스칼라값 리스트 계산

In [ ]:
sim_N = np.sum(sub_similarity_matrix, axis=1) + 1
print(sim_N.shape)

(610,)


검산

In [ ]:
# 첫번째 유저에 대한 값만 계산
sum(sub_similarity_matrix[0]) + 1

np.float64(1.6852889530733024)

In [ ]:
# 행렬로 한번에 계산한 첫번째 행
sim_N[0]

np.float64(1.6852889530733024)

In [ ]:
# 두 값 같다
sum(sub_similarity_matrix[0]) + 1 == sim_N[0]

np.True_

## Dot Production

모든 유저에 대한 평점 예측

행렬곱으로 한번에 계산

A = 유사도 행렬 (모든 유저 x 영화 본 유저)

B = 영화본 유저가 준 평점 행렬 (영화본 유저 x 1)

N = A의 유저별 유사도 총합값(스칼라) 리스트 (영화본 유저 개수, )


예측 = (A x B) / N

In [ ]:
# (610 x 3) * (3 x 1) / (610, )
pred_y = np.matmul(sub_similarity_matrix, watched_user_y).flatten() / sim_N
print(pred_y.shape)

(610,)


## 데이터프레임화

예측값 해석

Black Robe 영화에 대해
- 유저1은 1.39점 줬을 것
- 유저2는 0.54점 줬을 것
- 유저3은 0.20점 줬을 것
- 유저4는 1.15점 줬을 것
- ...

이렇게 계산된 예측값

데이터프레임화해서 추천할 때 사용할 것

In [ ]:
pred_y[:5]

array([1.38636218, 0.87716929, 1.19822725, 0.78183075, 1.04195781])

In [ ]:
all_users = list(user_similarity_df.index)
print(len(all_users))
print(all_users)

610
[1, 10, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 11, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 12, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 13, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 14, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 15, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 16, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 17, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 18, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 19, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 2, 20, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 21, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 22, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 23, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 24, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 25, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 26, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 27, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 28, 280, 281, 282

title 컬럼에 넣을 영화 리스트

모든 유저에 대한 Black Robe 영화

한 영화에 대한 예측이니

유저수만큼 늘린 리스트 필요

In [ ]:
title_list = [title] * len(all_users)
print(len(title_list))
print(title_list)

610
['Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', 'Black Robe', '

In [ ]:
cur_pred_df = pd.DataFrame(zip(title_list, all_users, pred_y),
                        columns=['title', 'userId', 'pred_rating'])
cur_pred_df

,title,userId,pred_rating
0,Black Robe,1,1.386362
1,Black Robe,10,0.877169
2,Black Robe,100,1.198227
3,Black Robe,101,0.781831
4,Black Robe,102,1.041958
...,...,...,...
605,Black Robe,95,1.327276
606,Black Robe,96,1.259578
607,Black Robe,97,0.962619
608,Black Robe,98,0.830475


# 평점 예측 모델

Dot Production 알고리즘


유사도 사용 전
- 영화m을 본 유저 U_m (um1, um2, um3)
- U_m이 영화m에 준 평점 평균은 (2+4+5) / 3 이지만,


유사도 사용 후

U_m과 유저u와의 유사한 정도로 평점 가중치 처리!
  - u에 대해 um1의 유사도0.5
  - u에 대해 um2의 유사도0.7
  - u에 대해 um3의 유사도0.8
- (2x0.5 + 4x0.7 + 5x0.8) / (0.5+0.7+0.8 + 1)




Dot Production 행렬화
- 유저 u 한명만 계산에서
- 한번에 모든 유저 U에 대해 계산
- 영화 m의 모든 유저에 대한 평점 예측
- 데이터 사이즈: |U|


for 문으로 모든 영화에 대해 계산하면
- 모든 영화마다
- 모든 유저에 대한
- 평점 예측 완료
- 데이터 사이즈: |M| * |U|

모든 사용자 별 모든 영화에 대해 주었을 평점에 대해 예측 계산됨

In [ ]:
from tqdm.notebook import tqdm

def modeling(user_similarity_matrix, movie_ratings_df):
    all_pred_df = pd.DataFrame()
    titles = sorted(movie_ratings_df['title'].unique())
    all_users = user_similarity_matrix.index
    n_users = len(all_users)

    for title in tqdm(titles):
        watched_user_indices = (
            movie_ratings_df[movie_ratings_df['title'] == title].index
        )

        # 유사도
        watched_user_ids_str = (
            movie_ratings_df.loc[watched_user_indices, 'user_id'].tolist()
        )
        watched_user_ids = [int(user_id) for user_id in watched_user_ids_str]

        sub_similarity_matrix = user_similarity_matrix.loc[watched_user_ids]
        sub_similarity_matrix = sub_similarity_matrix.T.to_numpy()
        sim_N = np.sum(sub_similarity_matrix, axis=1) + 1

        # 평점 예측
        watched_user_y = movie_ratings_df.loc[watched_user_indices, 'rating']
        watched_user_y = np.array(watched_user_y.tolist()).reshape(-1, 1)

        pred_y = np.matmul(sub_similarity_matrix, watched_user_y).flatten() / sim_N

        title_list = [title] * n_users
        cur_pred_df = pd.DataFrame(zip(title_list, all_users, pred_y),
                                columns=['title', 'user_id', 'pred_rating'])

        # 결과 기록
        all_pred_df = pd.concat([all_pred_df, cur_pred_df], axis=0)
    return all_pred_df

## 평점 예측

(약 14분 이상 소요)

In [ ]:
%%time
all_pred_df = modeling(user_similarity_df, movie_ratings_df)
all_pred_df

  0%|          | 0/9685 [00:00<?, ?it/s]

CPU times: user 4min 19s, sys: 3min 3s, total: 7min 22s
Wall time: 7min 25s


,title,user_id,pred_rating
0,'71 (2014),1,0.507529
1,'71 (2014),2,0.371641
2,'71 (2014),3,0.124477
3,'71 (2014),4,0.388859
4,'71 (2014),5,0.229232
...,...,...,...
605,À nous la liberté (Freedom for Us) (1931),606,0.126645
606,À nous la liberté (Freedom for Us) (1931),607,0.207591
607,À nous la liberté (Freedom for Us) (1931),608,0.155919
608,À nous la liberté (Freedom for Us) (1931),609,0.053546


## 평점 예측 결과 저장

In [ ]:
all_pred_df['user_id'] = all_pred_df['user_id'].astype(str)
all_pred_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5907850 entries, 0 to 609
Data columns (total 3 columns):
 #   Column       Dtype  
---  ------       -----  
 0   title        object 
 1   user_id      object 
 2   pred_rating  float64
dtypes: float64(1), object(2)
memory usage: 180.3+ MB


In [ ]:
all_pred_df.to_csv(DATA_DIR / "user_based_cf_prediction.csv", index=False)

# 예측 정확도 평가

테스트용 샘플 데이터 추출

In [ ]:
from sklearn.model_selection import train_test_split

_, test_data = train_test_split(
    movie_ratings_df[['user_id', 'title', 'rating']],
    test_size=20000, random_state=1234, stratify=movie_ratings_df['user_id'])
test_data

,user_id,title,rating
72424,469,Toy Story (1995),4.0
18649,119,"Lincoln Lawyer, The (2011)",4.0
21791,141,"Terminal, The (2004)",4.0
39110,271,Almost Famous (2000),3.5
65081,416,Ghostbusters (a.k.a. Ghost Busters) (1984),2.0
...,...,...,...
28716,199,There's Something About Mary (1998),5.0
70747,452,"Thomas Crown Affair, The (1999)",5.0
63059,414,Happiness (1998),4.0
24562,169,EuroTrip (2004),4.0


예측 평점 조인

In [ ]:
test_data = pd.merge(test_data, all_pred_df, on=['user_id', 'title'], how='left')
test_data

,user_id,title,rating,pred_rating
0,469,Toy Story (1995),4.0,3.809523
1,119,"Lincoln Lawyer, The (2011)",4.0,2.813754
2,141,"Terminal, The (2004)",4.0,3.182007
3,271,Almost Famous (2000),3.5,3.316803
4,416,Ghostbusters (a.k.a. Ghost Busters) (1984),2.0,3.430945
...,...,...,...,...
19995,199,There's Something About Mary (1998),5.0,3.546774
19996,452,"Thomas Crown Affair, The (1999)",5.0,3.442994
19997,414,Happiness (1998),4.0,3.327473
19998,169,EuroTrip (2004),4.0,2.612095


MAE, MSE, RMSE

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

true_y = np.array(test_data['rating'])
pred_y = np.array(test_data['pred_rating'])

mae = mean_absolute_error(y_true=true_y, y_pred=pred_y)
mse = mean_squared_error(y_true=true_y, y_pred=pred_y)
rmse = np.sqrt(mse)

print(f"MAE  : {str(round(mae, 2))}")
print(f"MSE  : {str(round(mse, 2))}")
print(f"RMSE : {str(round(rmse, 2))}")

MAE  : 0.84
MSE  : 1.06
RMSE : 1.03


Coverage

3점 이상인 경우 영화를 봤다고 가정 (혹은 추천했다고 가정)

In [ ]:
# 모델이 추천한 영화 개수
n_recommends = sum(1 * (pred_y > 4.0))
n_recommends

np.int64(1153)

In [ ]:
n_movies = movie_ratings_df['title'].nunique()

In [ ]:
# Coverage
n_recommends / n_movies

np.float64(0.11905007743933918)

Precision

In [ ]:
def get_precision(true_y, pred_y, threshold):
    trues = 1 * (true_y >= threshold)
    n_trues = sum(trues)

    pred_trues = 1 * (pred_y >= threshold)

    true_positive = sum(trues + pred_trues == 2)

    precision = true_positive / n_trues

    return precision

In [ ]:
get_precision(true_y, pred_y, 3)

np.float64(0.6399579909804164)

# 사용자 유사도 기반 추천

In [ ]:
# 샘플 사용자
user_id = "1"

In [ ]:
user_movie_df = movie_ratings_df[movie_ratings_df['user_id'] == user_id]
user_movie_df

,user_id,movie_id,rating,title
0,1,1,4.0,Toy Story (1995)
1,1,3,4.0,Grumpier Old Men (1995)
2,1,6,4.0,Heat (1995)
3,1,47,5.0,Seven (a.k.a. Se7en) (1995)
4,1,50,5.0,"Usual Suspects, The (1995)"
...,...,...,...,...
227,1,3744,4.0,Shaft (2000)
228,1,3793,5.0,X-Men (2000)
229,1,3809,4.0,What About Bob? (1991)
230,1,4006,4.0,Transformers: The Movie (1986)


In [ ]:
user_movie_pred_df = all_pred_df[all_pred_df['user_id'] == user_id]
user_movie_pred_df

,title,user_id,pred_rating
0,'71 (2014),1,0.507529
0,'Hellboy': The Seeds of Creation (2004),1,0.794120
0,'Round Midnight (1986),1,0.821511
0,'Salem's Lot (2004),1,0.263683
0,'Til There Was You (1997),1,0.439599
...,...,...,...
0,eXistenZ (1999),1,3.198649
0,xXx (2002),1,2.280353
0,xXx: State of the Union (2005),1,0.909248
0,¡Three Amigos! (1986),1,2.771625


In [ ]:
user_movie_df = pd.merge(user_movie_df, user_movie_pred_df, on=['user_id', 'title'], how='right')
user_movie_df

,user_id,movie_id,rating,title,pred_rating
0,1,NaN,NaN,'71 (2014),0.507529
1,1,NaN,NaN,'Hellboy': The Seeds of Creation (2004),0.794120
2,1,NaN,NaN,'Round Midnight (1986),0.821511
3,1,NaN,NaN,'Salem's Lot (2004),0.263683
4,1,NaN,NaN,'Til There Was You (1997),0.439599
...,...,...,...,...,...
9680,1,NaN,NaN,eXistenZ (1999),3.198649
9681,1,NaN,NaN,xXx (2002),2.280353
9682,1,NaN,NaN,xXx: State of the Union (2005),0.909248
9683,1,2478,4.0,¡Three Amigos! (1986),2.771625


In [ ]:
# 사용자가 아직 안 본 영화
movie_candidate_df = user_movie_df[user_movie_df['movie_id'].isnull()]
movie_candidate_df

,user_id,movie_id,rating,title,pred_rating
0,1,NaN,NaN,'71 (2014),0.507529
1,1,NaN,NaN,'Hellboy': The Seeds of Creation (2004),0.794120
2,1,NaN,NaN,'Round Midnight (1986),0.821511
3,1,NaN,NaN,'Salem's Lot (2004),0.263683
4,1,NaN,NaN,'Til There Was You (1997),0.439599
...,...,...,...,...,...
9679,1,NaN,NaN,anohana: The Flower We Saw That Day - The Movi...,0.017389
9680,1,NaN,NaN,eXistenZ (1999),3.198649
9681,1,NaN,NaN,xXx (2002),2.280353
9682,1,NaN,NaN,xXx: State of the Union (2005),0.909248


In [ ]:
# 예측 평점 기준 상위 10개
movie_candidate_df.sort_values(by='pred_rating', ascending=False)[:10]

,user_id,movie_id,rating,title,pred_rating
7571,1,NaN,NaN,"Shawshank Redemption, The (1994)",4.355593
3489,1,NaN,NaN,"Godfather, The (1972)",4.258401
3490,1,NaN,NaN,"Godfather: Part II, The (1974)",4.114438
2523,1,NaN,NaN,Dr. Strangelove or: How I Learned to Stop Worr...,4.096362
5554,1,NaN,NaN,Memento (2000),4.079292
2156,1,NaN,NaN,"Dark Knight, The (2008)",4.049551
1587,1,NaN,NaN,Casablanca (1942),4.049263
396,1,NaN,NaN,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",4.041298
5187,1,NaN,NaN,"Lord of the Rings: The Fellowship of the Ring,...",4.029114
2326,1,NaN,NaN,"Departed, The (2006)",4.010254


In [ ]:
# 사용자가 안본 영화 중 예측 평점 높은 10개 추천
def get_recomendation(
        movie_ratings_df,
        all_pred_df,
        user_id
    ):
    user_movie_df = movie_ratings_df[movie_ratings_df['user_id'] == user_id]
    user_movie_pred_df = all_pred_df[all_pred_df['user_id'] == user_id]

    user_movie_df = pd.merge(user_movie_df, user_movie_pred_df,
                             on=['user_id', 'title'], how='right')

    # 사용자가 아직 안 본 영화
    movie_candidate_df = user_movie_df[user_movie_df['movie_id'].isnull()]
    movie_candidate_df = movie_candidate_df.sort_values(by='pred_rating', ascending=False)[:10]

    return movie_candidate_df['title'].tolist()

In [ ]:
get_recomendation(movie_ratings_df, all_pred_df, user_id)

['Shawshank Redemption, The (1994)',
 'Godfather, The (1972)',
 'Godfather: Part II, The (1974)',
 'Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1964)',
 'Memento (2000)',
 'Dark Knight, The (2008)',
 'Casablanca (1942)',
 "Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)",
 'Lord of the Rings: The Fellowship of the Ring, The (2001)',
 'Departed, The (2006)']